# LangGraph + Repository Intelligence Orchestrator

This notebook composes the existing Repository Intelligence runtime with LangGraph.

**Important boundary:** LangGraph creates the plan and explicit step-role assignments. The existing `Orchestrator` executes the plan. It receives those assignments through `role_assignments=...`, so it does **not** run its role assigner a second time. The existing `QueryRouter` is injected into the orchestrator and continues to choose the concrete investigator specialist for each investigator step.

Before running, open this notebook with the `repository-intelligence` project environment active. The pinned subject snapshot and generated Architecture Context artifact must already exist.

## 1. Project path and imports

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from typing import Any, TypedDict
import json
from pathlib import Path
from IPython.display import display
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

# Prefer the current working directory when launched from the project root.
# Set REPO_INTELLIGENCE_ROOT if the notebook is launched from another directory.
PROJECT_ROOT = Path(
    os.environ.get("REPO_INTELLIGENCE_ROOT", str(Path.cwd()))
).expanduser().resolve()
PROJECT_ROOT = Path("C:/self-study/ai/build/repository-intelligence/")
SRC_ROOT = PROJECT_ROOT / "src"

if not (SRC_ROOT / "repo_intelligence").is_dir():
    raise FileNotFoundError(
        "Could not find src/repo_intelligence under PROJECT_ROOT="
        f"{PROJECT_ROOT}. Set REPO_INTELLIGENCE_ROOT to the repository root."
    )

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from repo_intelligence.llm.config import load_env
from repo_intelligence.llm.openai_client import OpenAILLMClient
from repo_intelligence.planning import (
    InvestigationPlan,
    build_planning_request,
    load_architecture_context,
    plan_from_response,
    validate_plan,
)
from repo_intelligence.orchestration import (
    Orchestrator,
    RoleAssignment,
    assign_roles,
)
from repo_intelligence.multiagent import TaskPayload
from repo_intelligence.agent import JsonFileTraceSink
from repo_intelligence.repository_query import (
    M17_ORCHESTRATION_CONFIG,
    OFFLINE_COMMIT,
    QueryRouter,
    build_architect,
    build_engine,
    build_flow_investigator,
    build_memory_context,
    build_memory_manager,
    build_visualizer,
    default_artifacts_root,
    default_subject_repository,
    live_bindings,
    read_snapshot_metadata,
    resolve_read_target,
    verify_subject_snapshot,
)

load_env()
print(f"Project root: {PROJECT_ROOT}")

## 2. Resolve and verify the pinned subject repository

In [ ]:
repository_root, readable_target, resolve_error = resolve_read_target()

if resolve_error:
    raise RuntimeError(f"Could not resolve subject repository: {resolve_error}")

verified_commit, commit_source, verification_error = verify_subject_snapshot(
    repository_root
)
if verification_error:
    raise RuntimeError(
        "Subject snapshot verification failed; refusing to analyze it: "
        f"{verification_error}"
    )

snapshot_metadata = read_snapshot_metadata(repository_root)
repository_label = (
    snapshot_metadata.get("name")
    or repository_root.name
    or str(repository_root)
)
current_commit = verified_commit or OFFLINE_COMMIT

print(f"Repository root: {repository_root}")
print(f"Repository scope: {repository_label}")
print(f"Commit scope: {current_commit}")
print(f"Commit source: {commit_source}")
print(f"Readable entry target: {readable_target}")

## 3. Load planner context and initialize the shared runtime

In [ ]:
# The planner consumes the existing Architecture Context artifact as input.
# It does not discover this artifact by calling repository tools.
architecture_context, architecture_context_record = load_architecture_context(
    output_root=default_artifacts_root()
)

memory_manager = build_memory_manager()
memory_context = build_memory_context(
    repository=repository_label,
    current_commit=current_commit,
)
context_engine = build_engine(repository_label)
trace_sink = JsonFileTraceSink()

tool_bindings = live_bindings(
    repository_root,
    output_root=default_artifacts_root(),
)

print("Architecture Context:", architecture_context_record)
print("Shared runtime initialized.")

## 4. Build the specialist agents, QueryRouter, and existing Orchestrator

In [ ]:
# Separate clients make the specialist responsibilities explicit.
planner_llm = OpenAILLMClient()
architect_llm = OpenAILLMClient()
flow_llm = OpenAILLMClient()
visualizer_llm = OpenAILLMClient()

architect = build_architect(
    llm=architect_llm,
    trace_sink=trace_sink,
    memory_manager=memory_manager,
    memory_context=memory_context,
    context_engine=context_engine,
    bindings=tool_bindings,
)

flow_investigator = build_flow_investigator(
    llm=flow_llm,
    trace_sink=trace_sink,
    memory_manager=memory_manager,
    memory_context=memory_context,
    context_engine=context_engine,
    bindings=tool_bindings,
)

visualizer = build_visualizer(
    llm=visualizer_llm,
    trace_sink=trace_sink,
    memory_manager=memory_manager,
    memory_context=memory_context,
    context_engine=context_engine,
)

# The same router instance is used for route preview and actual execution.
router = QueryRouter(
    architecture=architect,
    flow=flow_investigator,
)

orchestrator = Orchestrator(
    investigator=router,
    analyst=visualizer,
    trace_sink=trace_sink,
    config=M17_ORCHESTRATION_CONFIG,
)

print("Architecture investigator:", type(architect).__name__)
print("Flow investigator:", type(flow_investigator).__name__)
print("Analyst:", type(visualizer).__name__)
print("QueryRouter and Orchestrator initialized.")

## 5. Define the LangGraph state

In [ ]:
class AgentAssignments(BaseModel):
    """Readable preview: plan step ID -> concrete specialist slot."""

    agent_assignments: dict[str, str] = Field(default_factory=dict)


class RepoIntelligenceState(TypedDict, total=False):
    objective: str
    plan: InvestigationPlan
    role_assignment: list[RoleAssignment]
    agent_assignments: AgentAssignments
    orchestration_run: Any
    execution_status: str
    execution_error: str

## 6. Planner node

In [ ]:
def planner_node(state: RepoIntelligenceState) -> dict:
    objective = (state.get("objective") or "").strip()
    if not objective:
        raise ValueError("state['objective'] must be a non-empty string")

    request = build_planning_request(
        objective,
        architecture_context=architecture_context,
    )
    response = planner_llm.complete(request)
    plan = plan_from_response(response)

    # Semantic validation remains deterministic and application-owned.
    validate_plan(plan)

    print(f"Plan created: {len(plan.steps)} step(s)")
    for step in plan.steps:
        print(f"  {step.id}: {step.goal}")

    return {"plan": plan}

## 7. Role assignment node

In [ ]:
def role_assigner_node(state: RepoIntelligenceState) -> dict:
    plan = state["plan"]
    assignments = assign_roles(plan.steps)

    # Validate exact coverage here so the failure is clearly attributed to
    # the LangGraph assignment stage, before orchestration begins.
    step_ids = [step.id for step in plan.steps]
    assignment_ids = [item.step_id for item in assignments]

    if len(step_ids) != len(set(step_ids)):
        raise ValueError("The plan contains duplicate step IDs.")
    if len(assignment_ids) != len(set(assignment_ids)):
        raise ValueError("Role assignment contains duplicate step IDs.")
    if set(assignment_ids) != set(step_ids):
        raise ValueError(
            "Role assignments must exactly cover plan step IDs. "
            f"Plan={step_ids}; assignments={assignment_ids}"
        )

    print("Preassigned roles:")
    for item in assignments:
        print(f"  {item.step_id}: {item.role} — {item.reason}")

    return {"role_assignment": assignments}

## 8. Query routing preview node

In [ ]:
def query_router_node(state: RepoIntelligenceState) -> dict:
    """Preview concrete routing; execution still uses the injected router.

    The Orchestrator accepts role assignments (INVESTIGATOR / ANALYST),
    not concrete agent-name assignments. It uses this same `router` object
    for actual investigator dispatch during `execute`.
    """
    plan = state["plan"]
    role_assignments = state["role_assignment"]

    step_ids = [step.id for step in plan.steps]
    if len(step_ids) != len(set(step_ids)):
        raise ValueError("The plan contains duplicate step IDs.")

    role_by_step: dict[str, RoleAssignment] = {}
    for assignment in role_assignments:
        if assignment.step_id in role_by_step:
            raise ValueError(
                f"Duplicate role assignment for step {assignment.step_id!r}"
            )
        role_by_step[assignment.step_id] = assignment

    if set(role_by_step) != set(step_ids):
        raise ValueError(
            "Role assignments must exactly cover plan step IDs. "
            f"Plan={step_ids}; assignments={list(role_by_step)}"
        )

    agent_names: dict[str, str] = {}
    for step in plan.steps:
        role = role_by_step[step.id].role

        if role == "ANALYST":
            agent_name = "visualizer"
        elif role == "INVESTIGATOR":
            specialist = router._resolve(step.id, step.goal)
            if specialist is architect:
                agent_name = "architect"
            elif specialist is flow_investigator:
                agent_name = "flow_investigator"
            else:
                raise ValueError(
                    f"QueryRouter returned an unexpected specialist for "
                    f"step {step.id!r}: {type(specialist).__name__}"
                )
        else:
            raise ValueError(
                f"Unsupported role {role!r} for step {step.id!r}"
            )

        agent_names[step.id] = agent_name
        print(f"  {step.id}: {role} -> {agent_name}")

    return {
        "agent_assignments": AgentAssignments(
            agent_assignments=agent_names
        )
    }

## 9. Orchestrator node — pass the preassigned roles

In [ ]:
def orchestrator_node(state: RepoIntelligenceState) -> dict:
    """Execute through the existing orchestration runtime.

    Critical integration: `role_assignments=...` tells the existing
    Orchestrator to validate and use the roles produced by LangGraph,
    bypassing its internal role assigner.
    """
    task = TaskPayload(
        question=state["objective"],
        repository_scope=repository_label,
        commit_scope=current_commit,
    )

    run = orchestrator.execute(
        task,
        state["plan"],
        role_assignments=state["role_assignment"],
    )

    print(f"Orchestration status: {run.status}")
    print(f"Termination reason: {run.termination_reason}")
    print(f"Completed steps: {run.completed_steps}")
    print(f"Failed steps: {run.failed_steps}")
    print(f"Blocked steps: {run.blocked_steps}")

    return {
        "orchestration_run": run,
        "execution_status": run.status,
        "execution_error": run.termination_reason or "",
    }

## 10. Build and compile the LangGraph workflow

In [ ]:
builder = StateGraph(RepoIntelligenceState)

builder.add_node("planner", planner_node)
builder.add_node("assign_role", role_assigner_node)
builder.add_node("query_router", query_router_node)
builder.add_node("orchestrator", orchestrator_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "assign_role")
builder.add_edge("assign_role", "query_router")
builder.add_edge("query_router", "orchestrator")
builder.add_edge("orchestrator", END)

graph = builder.compile()
print(graph.get_graph().draw_mermaid())

## 11. Invoke the workflow and inspect the result

In [ ]:
objective = "How do agents and MCP servers interact?"

result = graph.invoke({"objective": objective})

print("\n==================== FINAL RUN SUMMARY ====================")
print("Status:", result.get("execution_status"))
print("Termination:", result.get("execution_error"))

run = result.get("orchestration_run")

if run is not None:
    print("Run ID:", run.run_id)
    print("Final:", run.final)
    print("Completed steps:", run.completed_steps)
    print("Failed steps:", run.failed_steps)
    print("Blocked steps:", run.blocked_steps)

    if run.finding is not None:
        print("\nFinding:")
        print(run.finding)

    if run.report is not None:
        print("\nReport:")
        print(run.report)

    print("\nRole assignments:")
    for assignment in result.get("role_assignment", []):
        print(
            f"  {assignment.step_id}: "
            f"{assignment.role} — {assignment.reason}"
        )

    print("\nConcrete routing preview:")
    print(result.get("agent_assignments"))

    # -----------------------------------------------------
    # Agent execution trace summary
    # -----------------------------------------------------
    print("\n==================== AGENT EXECUTION TRACES ====================")

    trace_root = Path(f"{PROJECT_ROOT}/runtime/agent_runs")

    if not trace_root.exists():
        print(f"Trace directory not found: {trace_root.resolve()}")
    else:
        trace_files = sorted(trace_root.glob("*/trace.json"))

        if not trace_files:
            print(f"No trace.json files found under {trace_root.resolve()}")

        for trace_path in trace_files:
            try:
                with trace_path.open(encoding="utf-8") as f:
                    trace = json.load(f)
            except (OSError, json.JSONDecodeError) as exc:
                print(f"Could not read {trace_path}: {exc}")
                continue

            events = trace.get("orchestration_events", [])
            summary = trace.get("summary", {})

            print(f"\nTrace file: {trace_path}")
            print("Trace ID:", trace.get("run_id"))
            print("Started:", trace.get("started_at"))
            print("Completed:", trace.get("completed_at"))
            print("Tool calls executed:", summary.get("tool_calls_executed"))
            print("Tool calls blocked:", summary.get("tool_calls_blocked"))

            for event in events:
                print(
                    " ",
                    event.get("timestamp"),
                    event.get("operation"),
                    "step=", event.get("step_id"),
                    "role=", event.get("agent_role"),
                    "status=", event.get("status"),
                )
